In [0]:
# Healthcare Data Quality Framework

from pyspark.sql import functions as F
from functools import reduce

catalog_schema = "workspace.healthcare"

print("Healthcare Data Quality Checks Started")

In [0]:
# Load Silver tables

patients_df = spark.table(f"{catalog_schema}.silver_patients")
doctors_df = spark.table(f"{catalog_schema}.silver_doctors")
appointments_df = spark.table(f"{catalog_schema}.silver_appointments")
billing_df = spark.table(f"{catalog_schema}.silver_billing")

print("All Silver tables loaded successfully")

In [0]:
# Store all data quality results here

quality_results = []

def run_quality_check(table_name, check_name, total_records, invalid_records):

    status = "PASS" if invalid_records == 0 else "FAIL"

    quality_results.append((
        table_name,
        check_name,
        int(total_records),
        int(invalid_records),
        status
    ))

    print(
        f"{table_name} | {check_name} | "
        f"Invalid: {invalid_records} | Status: {status}"
    )

In [0]:
# Patient quality checks

patient_count = patients_df.count()

patient_null_condition = reduce(
    lambda a, b: a | b,
    [F.col(c).isNull() for c in [
        "PatientID", "PatientName", "Age", "Gender", "City"
    ]]
)

patient_nulls = patients_df.filter(patient_null_condition).count()

patient_duplicates = (
    patient_count -
    patients_df.select("PatientID").distinct().count()
)

invalid_patient_age = (
    patients_df
    .filter((F.col("Age") < 0) | (F.col("Age") > 120))
    .count()
)

run_quality_check(
    "silver_patients", "Null Value Check",
    patient_count, patient_nulls
)

run_quality_check(
    "silver_patients", "Duplicate PatientID Check",
    patient_count, patient_duplicates
)

run_quality_check(
    "silver_patients", "Invalid Age Check",
    patient_count, invalid_patient_age
)


# Doctor quality checks

doctor_count = doctors_df.count()

doctor_null_condition = reduce(
    lambda a, b: a | b,
    [F.col(c).isNull() for c in [
        "DoctorID", "DoctorName", "Department", "Experience"
    ]]
)

doctor_nulls = doctors_df.filter(doctor_null_condition).count()

doctor_duplicates = (
    doctor_count -
    doctors_df.select("DoctorID").distinct().count()
)

invalid_experience = (
    doctors_df
    .filter(F.col("Experience") < 0)
    .count()
)

run_quality_check(
    "silver_doctors", "Null Value Check",
    doctor_count, doctor_nulls
)

run_quality_check(
    "silver_doctors", "Duplicate DoctorID Check",
    doctor_count, doctor_duplicates
)

run_quality_check(
    "silver_doctors", "Invalid Experience Check",
    doctor_count, invalid_experience
)

In [0]:
# Appointment quality checks

appointment_count = appointments_df.count()

appointment_null_condition = reduce(
    lambda a, b: a | b,
    [F.col(c).isNull() for c in [
        "AppointmentID",
        "PatientID",
        "DoctorID",
        "AppointmentDate",
        "Status"
    ]]
)

appointment_nulls = (
    appointments_df
    .filter(appointment_null_condition)
    .count()
)

appointment_duplicates = (
    appointment_count -
    appointments_df.select("AppointmentID").distinct().count()
)

invalid_appointment_status = (
    appointments_df
    .filter(
        ~F.col("Status").isin("Completed", "Pending", "Cancelled")
    )
    .count()
)

invalid_patient_reference = (
    appointments_df
    .join(
        patients_df.select("PatientID"),
        on="PatientID",
        how="left_anti"
    )
    .count()
)

invalid_doctor_reference = (
    appointments_df
    .join(
        doctors_df.select("DoctorID"),
        on="DoctorID",
        how="left_anti"
    )
    .count()
)

run_quality_check(
    "silver_appointments", "Null Value Check",
    appointment_count, appointment_nulls
)

run_quality_check(
    "silver_appointments", "Duplicate AppointmentID Check",
    appointment_count, appointment_duplicates
)

run_quality_check(
    "silver_appointments", "Invalid Status Check",
    appointment_count, invalid_appointment_status
)

run_quality_check(
    "silver_appointments", "Patient Reference Check",
    appointment_count, invalid_patient_reference
)

run_quality_check(
    "silver_appointments", "Doctor Reference Check",
    appointment_count, invalid_doctor_reference
)


# Billing quality checks

billing_count = billing_df.count()

billing_null_condition = reduce(
    lambda a, b: a | b,
    [F.col(c).isNull() for c in [
        "BillingID", "AppointmentID", "Amount", "PaymentStatus"
    ]]
)

billing_nulls = billing_df.filter(billing_null_condition).count()

billing_duplicates = (
    billing_count -
    billing_df.select("BillingID").distinct().count()
)

invalid_billing_amount = (
    billing_df
    .filter(F.col("Amount") < 0)
    .count()
)

invalid_payment_status = (
    billing_df
    .filter(
        ~F.col("PaymentStatus").isin("Paid", "Pending", "Cancelled")
    )
    .count()
)

invalid_appointment_reference = (
    billing_df
    .join(
        appointments_df.select("AppointmentID"),
        on="AppointmentID",
        how="left_anti"
    )
    .count()
)

run_quality_check(
    "silver_billing", "Null Value Check",
    billing_count, billing_nulls
)

run_quality_check(
    "silver_billing", "Duplicate BillingID Check",
    billing_count, billing_duplicates
)

run_quality_check(
    "silver_billing", "Invalid Billing Amount Check",
    billing_count, invalid_billing_amount
)

run_quality_check(
    "silver_billing", "Invalid Payment Status Check",
    billing_count, invalid_payment_status
)

run_quality_check(
    "silver_billing", "Appointment Reference Check",
    billing_count, invalid_appointment_reference
)

In [0]:
# Create Data Quality results DataFrame

result_schema = [
    "TableName",
    "CheckName",
    "TotalRecords",
    "InvalidRecords",
    "Status"
]

quality_df = spark.createDataFrame(
    quality_results,
    result_schema
)

quality_df = quality_df.withColumn(
    "CheckTimestamp",
    F.current_timestamp()
)

display(quality_df)

In [0]:
# Save Data Quality audit results

(
    quality_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{catalog_schema}.dq_audit_results")
)

print("Data Quality results saved successfully!")

In [0]:
# Read saved quality audit results

audit_df = spark.table(
    f"{catalog_schema}.dq_audit_results"
)

display(
    audit_df.orderBy(F.desc("CheckTimestamp"))
)

# Summary of current notebook execution

passed_checks = sum(
    1 for result in quality_results
    if result[4] == "PASS"
)

failed_checks = sum(
    1 for result in quality_results
    if result[4] == "FAIL"
)

print("========== DATA QUALITY SUMMARY ==========")
print("Total Checks:", len(quality_results))
print("Passed Checks:", passed_checks)
print("Failed Checks:", failed_checks)

if failed_checks == 0:
    print("Overall Result: ALL CHECKS PASSED")
else:
    print("Overall Result: SOME CHECKS FAILED")